In [ ]:
from option_chain_downloader import OptionChainDownloader
from option_finder import *
from option_data_plotter import *

In [ ]:
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/all_options.log')

chain_dir = 'chain'
quotes_dir = 'quotes'
data_dir = 'data'
cookie_file = 'cookie.txt'
ocd = OptionChainDownloader(chain_dir, quotes_dir, cookie_file, logger, strikes='ALL')
self = OptionFinder(logger, chain_dir=chain_dir, report_dir=data_dir)

In [ ]:
symlist  = ['QQQ', 'SPY', 'TSM', 'GOOGL']
symlist += ['DIA', 'IBIT']
symlist += ['GLD', 'TLT', 'CRCL', 'NVDA', 'AAPL', 'MSFT']
symlist += ['TSLA', 'PLTR', 'META', 'AMZN', 'AVGO', 'SMH', 'ETHA']

## Refresh data here

In [ ]:
if os.path.exists('cookie.txt.expired'):
    if os.path.getmtime('cookie.txt') > os.path.getmtime('cookie.txt.expired'):
        os.unlink('cookie.txt.expired')
        print('expired cookie file removed')
    else:
        print('cookie file expired')

In [ ]:
_t0 = time.time()
_n = ocd.download_option_chain(symlist, batch_size=5, rps=5)
print(_n, 'file downloads requested in', int(time.time() - _t0), 'seconds')

In [ ]:
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
_t0 = time.time()
_df = self.build_option_df(symlist)
_t1 = time.time()
print(f'build_option_df {_t1 - _t0:.1f} seconds')
dfcp = self.concat_put_call_options(_df)
dfcp = bucketize_dte(add_moneyness_columns(dfcp))
_t2 = time.time()
print(f'concat_put_call_options {_t2 - _t1:.1f} seconds')
px.bar(check_data_age(_df), y=['load_age', 'quote_age'], barmode='group', title=f"Data Ages", width=800, height=300).show()
print(f'px.bar {time.time() - _t2:.1f} seconds')
dfcp.loc[:, ['dte', 'expDt']].groupby('dte').first().head(24).tail(20).T

In [ ]:
print('Earning dates:', count_days_from_earning_reports(df_quotes)['earningDays'].to_dict())

In [ ]:
_dte = 45
print(f'DTE closest to {_dte} is', get_closest_value_in_column(dfcp, 'dte', _dte))

### ImpVola Overview

In [ ]:
plot_iv_statistics(dfcp)

In [ ]:
_symbol = 'DIA'
_dte = 159
_strike = 605
_strike = get_closest_value_in_column(dfcp[dfcp.symbol==_symbol], 'strike', _strike)
#px.scatter(get_theta_curves(dfcp, _symbol, _strike), title=f'Theta curves of {_symbol} strike {_strike}', height=500)
_df_thc = get_theta_curve(dfcp[dfcp.dte <= _dte], _symbol, _strike, 'C')
_df_thc

In [ ]:
dfcp[(dfcp.symbol==_symbol)&(dfcp.dte==_dte)&(dfcp.strike==_strike)&(dfcp.type=='C')]['mid']

In [ ]:
prepare_theta_curve(_df_thc, _dte, 1.15).dropna()

In [ ]:
compute_time_decay_metrics(dfcp, _symbol, 'C', _dte, _strike, debug=True)

### Compute time decay metrics for CALL options with some open interests

In [ ]:
_lodfc = [compute_all_time_decay_metrics_for_symbol(dfcp, symbol, 'C', ignore_no_bid=False, oi_lb=100) for symbol in symlist]
_dfc = pd.concat(_lodfc)
_dfc['dthr'] = _dfc.dth/_dfc.dte
_dfc['dtzr'] = _dfc.dtz/_dfc.dte
_index = ['symbol', 'dte', 'strike']
add_cols = ['lastPrice', 'Delta', 'Theta', 'overpaid', 'leverage', 'expDt', 'OpenInterest', 'pctSpread']
dfc = _dfc.set_index(_index).join(dfcp[dfcp.type=='C'].set_index(_index).loc[:, add_cols]).reset_index()

### Something is wrong with hdte_resid < resid

In [ ]:
dfc[(dfc.hdte_resid < dfc.resid) & (np.abs(dfc.hdte_resid - dfc.resid) > 1e-7)].head(60)

In [ ]:
_dfc = dfc[(dfc.symbol!='TLT')&(dfc.dte >= 90) & (dfc.hdte_resid >= 0.999) & (dfc.overpaid <= 0.03) & (dfc.pctSpread <= 2)].sort_values(by='leverage', ascending=False)
_dfc.head()

In [ ]:
px.scatter(_dfc.head(160), x='hdte_resid', y='leverage', color='symbol', height=600)

## Puts

In [ ]:
_lodf = [compute_all_time_decay_metrics_for_symbol(dfcp, symbol, 'P', ignore_no_bid=False, oi_lb=100) for symbol in symlist]
_dfp = pd.concat(_lodf)
_dfp['dthr'] = _dfp.dth/_dfp.dte
_dfp['dtzr'] = _dfp.dtz/_dfp.dte
_index = ['symbol', 'dte', 'strike']
add_cols = ['Delta', 'pctSpread', 'lastPrice', 'Theta', 'moneyness', 'expDt', 'OpenInterest']
dfp = _dfp.set_index(_index).join(dfcp[dfcp.type=='P'].set_index(_index).loc[:, add_cols]).reset_index()
dfp['pctProfit'] = dfp.premium/2/dfp.strike/dfp.dth*100*365

### hdte_resid should not be less than resid

In [ ]:
dfp[dfp.hdte_resid < dfp.resid]

In [ ]:
_dfp = dfp[(dfp.strike <= 0.999*dfp.lastPrice) & (dfp.pctSpread <= 5) & (dfp.dthr <= 0.5)].sort_values(by='pctProfit', ascending=False)
print(_dfp.shape)
_dfp.head()

In [ ]:
px.scatter(_dfp.head(1000), x='Delta', y='pctProfit', color='symbol', height=600)

In [ ]:
dfl = plot_leverage_overpaid(dfcp[(dfcp.dte >= 90) & (dfcp.dte <= 360)], delta_lb=0.5, overpaid_ub=0.05, price_lb=5, spread_ub=5, leverage_lb=2, openinterest_lb=100)
dfl = dfl.drop(columns=['pctProfit', 'type', 'Rho', 'cluster', 'dte_cluster'])

In [ ]:
_filter = (dfl.index.get_level_values(0) == 'GOOGL') & (dfl.dtzr >= 10)
dfl[_filter].sort_values(by='leverage', ascending=False).head(60)

### Put Debit Spread

In [ ]:
def calc_pds_debit(df):
    strike0 = df.strike.iloc[0]
    spreads = list(df.strike.iloc[[1, -1]] - strike0)
    return int((df.mid.iloc[0]*2 + df.mid.iloc[1] - df.mid.iloc[-1])*100)/100, strike0, spreads

def get_final_dfpds(dfpds):
    pds_list = []
    for idx in dfpds.index:
        _df = put_debit_spread(dfpds, *idx, dfcp)
        debit, strike0, spreads = calc_pds_debit(_df)
        pds_list.append({'symbol': idx[0], 'dte': idx[1], 'debit': debit, 'strike0': strike0, 'spread1': spreads[0], 'spread2': spreads[1]})
    return dfpds.join(pd.DataFrame(pds_list, index=dfpds.index))

In [ ]:
dfpds = select_pds_deltas(dfcp, 35, 45)
dfpds

## Total open interests and volumes for all dte and strikes

In [ ]:
_df = dfcp.loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False)

### It may be of interest to look at OpenInterests and Volumes for 8-weeks and 1-year DTE clusters

In [ ]:
for _dte_cluster in ['8wk', '1yr']:
    _df = dfcp[dfcp.dte_cluster==_dte_cluster].loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
    plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False, log_y_threshold=500, horizontal_spacing=0.03)

In [ ]:
_df = calc_overall_put_call_ratios(dfcp)
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_df = calc_overall_put_call_ratios(dfcp[dfcp.cluster=='atm'])
_df = _df.rename(columns=dict([(c, 'atm '+c) for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_dte_lb = 90
_dte_ub = 180
_df = calc_overall_put_call_ratios(dfcp[(dfcp.cluster=='atm') & (dfcp.dte >= _dte_lb) & (dfcp.dte <= _dte_ub)])
_df = _df.rename(columns=dict([(c, f'atm dte {_dte_lb} to {_dte_ub} {c}') for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:], shared_y=False)

### The End